# QuantIQ | Phase 2 — Market Analysis

**Phase:** 2 | **Weeks:** 4–6 | **Deadline:** 14 June 2026

**Purpose:** Individual stock analysis for 12 NIFTY50 stocks — technical indicators, fundamentals, and cross-sector correlation.

---

## Team

| Handle | Stock          | Sector                |
|--------|----------------|-----------------------|
| AJ     | RELIANCE.NS    | Energy / Conglomerate |
| AV     | TCS.NS         | IT                    |
| AK     | INFY.NS        | IT                    |
| SS     | HCLTECH.NS     | IT                    |
| AR     | HDFCBANK.NS    | Banking               |
| EB     | ICICIBANK.NS   | Banking               |
| ShS    | AXISBANK.NS    | Banking               |
| RS     | TVSMOTOR.NS         | Auto                  |
| GT     | M&M.NS         | Auto                  |
| RT     | LT.NS          | Infrastructure        |
| NS     | TITAN.NS       | Consumer              |
| SmS    | APOLLOMICRO.NS | Defence               |

**Section 13 — Correlation Heatmap:** RS + GT

---

*Created by RS (Project Lead). Do not modify header cells or shared helper cells.*

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import yfinance as yf
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from src.data import fetch_ohlc

pio.templates["plotly_dark"].layout.paper_bgcolor = "#252526"
pio.templates["plotly_dark"].layout.plot_bgcolor  = "#1F2531"
pio.templates.default = "plotly_dark"

print("Imports OK")

Imports OK


In [2]:
TICKERS = {
    "AJ":  "RELIANCE.NS",
    "AV":  "TCS.NS",
    "AK":  "INFY.NS",
    "SS":  "HCLTECH.NS",
    "AR":  "HDFCBANK.NS",
    "EB":  "ICICIBANK.NS",
    "ShS": "AXISBANK.NS",
    "RS":  "TVSMOTOR.NS",
    "GT":  "M&M.NS",
    "RT":  "LT.NS",
    "NS":  "TITAN.NS",
    "SmS": "APOLLOMICRO.NS",
}

PERIOD   = "1y"
INTERVAL = "1d"

print(f"Config loaded — {len(TICKERS)} tickers, period={PERIOD}, interval={INTERVAL}")

Config loaded — 12 tickers, period=1y, interval=1d


In [3]:
def fetch_stock(
    ticker: str,
    period: str = PERIOD,
    interval: str = INTERVAL,
) -> pd.DataFrame:
    """Fetch OHLCV for a .NS ticker via fetch_ohlc.

    Args:
        ticker (str): NSE ticker with .NS suffix (e.g. "RELIANCE.NS").
        period (str): yfinance period string. Default "1y".
        interval (str): yfinance interval string. Default "1d".

    Returns:
        pd.DataFrame: OHLCV DataFrame indexed by datetime, NaN rows dropped.

    Raises:
        AssertionError: If ticker does not end with .NS.
        ValueError: If no data returned for ticker.
    """
    assert ticker.endswith(".NS"), f"Ticker must have .NS suffix, got: {ticker}"
    df = fetch_ohlc(ticker, period=period, interval=interval, use_cache=True)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    return df.dropna()

## Energy

## Section 1 — RELIANCE.NS | AJ (Energy / Conglomerate)

In [4]:
# ── 1a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

RELIANCE.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,1295.012426,1305.165434,1287.148760,1297.699951,23474327
2026-06-05,1304.500000,1306.000000,1288.000000,1291.000000,17785223
2026-06-08,1277.000000,1282.599976,1259.199951,1263.300049,16494759
2026-06-09,1269.000000,1274.199951,1257.500000,1269.199951,23620214
2026-06-10,1275.000000,1300.500000,1254.400024,1258.800049,21003689


In [5]:
# ── 1b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1422.76,1434.45,1410.69,1421.73,13107834.82
std,71.24,70.57,71.90,71.39,6916074.78
min,1269.00,1274.20,1254.40,1258.80,0.00
25%,1371.01,1381.65,1360.53,1370.07,8288419.00
50%,1410.10,1420.24,1395.50,1406.55,11215789.00
75%,1476.47,1485.73,1463.01,1472.83,16992539.75
max,1585.67,1604.38,1570.94,1584.97,42634247.00


In [6]:
# ── 1c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [7]:
# ── 1d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [8]:
# ── 1e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [9]:
# ── 1f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [10]:
# ── 1g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 21.10
  EPS Growth (TTM) : -12.60%
  Debt / Equity    : 36.65
  Total Cash       : ₹2,581,470,117,888
  Total Debt       : ₹3,979,999,969,280
  Current Ratio    : 1.10
  PEG Ratio        : 0.82
  Dividend Yield   : 47.00%


### Observations

_TODO (AJ): Write 3–5 paragraphs on what the technical and fundamental data reveals about RELIANCE.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## IT

## Section 2 — TCS.NS | AV (IT)

In [11]:
# ── 2a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TCS.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,2241.699951,2253.100098,2216.600098,2241.000000,4867263
2026-06-05,2262.899902,2271.800049,2192.000000,2198.899902,5019717
2026-06-08,2170.000000,2177.000000,2143.300049,2151.399902,6536734
2026-06-09,2174.000000,2174.000000,2132.800049,2151.000000,4148655
2026-06-10,2151.300049,2180.800049,2144.800049,2153.899902,2847680


In [12]:
# ── 2b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,2858.04,2879.91,2829.64,2851.95,3425032.34
std,326.11,323.40,326.88,326.93,2116289.53
min,2151.30,2174.00,2132.80,2151.00,0.00
25%,2539.57,2571.26,2522.23,2544.50,2289789.00
50%,2951.57,2971.82,2929.71,2950.88,2905922.00
75%,3103.12,3122.38,3080.30,3097.02,3931979.25
max,3385.87,3404.05,3356.04,3382.21,16331582.00


In [13]:
# ── 2c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [14]:
# ── 2d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [15]:
# ── 2e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [16]:
# ── 2f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [17]:
# ── 2g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.84
  EPS Growth (TTM) : 12.20%
  Debt / Equity    : 10.39
  Total Cash       : ₹410,799,996,928
  Total Debt       : ₹112,829,997,056
  Current Ratio    : 2.23
  PEG Ratio        : 2.33
  Dividend Yield   : 576.00%


### Observations

_TODO (AV): Write 3–5 paragraphs on what the technical and fundamental data reveals about TCS.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 3 — INFY.NS | AK (IT)

In [18]:
# ── 3a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

INFY.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,1182.413264,1189.460712,1170.863152,1175.855225,12755957
2026-06-05,1194.159169,1197.878729,1169.003569,1172.135742,8692290
2026-06-08,1152.559265,1175.267816,1151.580447,1162.445312,8631711
2026-06-09,1168.318269,1168.318269,1142.575403,1155.300049,10984054
2026-06-10,1160.099976,1171.400024,1143.300049,1145.300049,8885958


In [19]:
# ── 3b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1415.22,1428.48,1400.46,1413.66,10140542.83
std,152.47,151.06,151.85,152.38,7223530.64
min,1078.66,1098.14,1065.93,1071.81,0.00
25%,1279.54,1295.88,1266.27,1279.95,6234342.00
50%,1449.63,1459.70,1433.32,1445.90,8388425.00
75%,1534.86,1552.34,1523.29,1544.38,12088294.75
max,1690.62,1691.40,1631.20,1654.01,69085430.00


In [20]:
# ── 3c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [21]:
# ── 3d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [22]:
# ── 3e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [23]:
# ── 3f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [24]:
# ── 3g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.07
  EPS Growth (TTM) : 11.80%
  Debt / Equity    : 9.83
  Total Cash       : ₹3,705,999,872
  Total Debt       : ₹967,000,000
  Current Ratio    : 1.98
  PEG Ratio        : 2.21
  Dividend Yield   : 424.00%


### Observations

_TODO (AK): Write 3–5 paragraphs on what the technical and fundamental data reveals about INFY.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 4 — HCLTECH.NS | SS (IT)

In [25]:
# ── 4a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HCLTECH.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,1166.400024,1176.5,1158.199951,1168.300049,2025595
2026-06-05,1179.000000,1182.5,1147.000000,1154.699951,1971395
2026-06-08,1141.900024,1159.0,1132.300049,1151.300049,1546444
2026-06-09,1158.699951,1159.0,1136.400024,1146.300049,2225582
2026-06-10,1140.800049,1152.0,1130.000000,1132.099976,1600150


In [26]:
# ── 4b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1453.16,1467.12,1437.32,1451.49,3190956.68
std,152.46,152.01,151.31,152.88,2725274.52
min,1121.90,1143.20,1103.40,1124.00,0.00
25%,1363.37,1379.04,1338.02,1353.07,1938355.00
50%,1443.09,1456.43,1428.78,1443.53,2566080.00
75%,1589.53,1610.73,1580.25,1594.84,3456803.25
max,1746.56,1746.66,1668.56,1697.11,33066256.00


In [27]:
# ── 4c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [28]:
# ── 4d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [29]:
# ── 4e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [30]:
# ── 4f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [31]:
# ── 4g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 18.46
  EPS Growth (TTM) : -0.20%
  Debt / Equity    : 6.93
  Total Cash       : ₹3,204,999,936
  Total Debt       : ₹550,000,000
  Current Ratio    : 2.22
  PEG Ratio        : 2.39
  Dividend Yield   : 837.00%


# Observations

## 1. Code Observations

The notebook successfully fetched one year of HCLTECH.NS market data consisting of 250 trading sessions and generated multiple technical indicators including EMA20, EMA50, MACD (12,26,9), RSI (14), volume trends, and fundamental metrics. The code is organized into independent sections that progressively build the analysis from raw OHLCV data to technical and fundamental evaluation.

The EMA analysis correctly overlays short-term and medium-term moving averages on the candlestick chart, enabling trend identification. The MACD implementation calculates the MACD line, signal line, and histogram, providing momentum analysis, while the RSI implementation highlights overbought and oversold zones using the standard 70/30 thresholds. Volume analysis is enhanced through a 20-day rolling average, making unusual trading activity easier to identify.

The notebook also retrieves company fundamentals through Yahoo Finance and formats key metrics such as P/E ratio, PEG ratio, debt-to-equity ratio, cash position, debt levels, current ratio, and dividend yield. Together, these modules provide a comprehensive framework for combining technical analysis with fundamental analysis.

## 2. Market and Financial Observations

### Trend Direction

HCLTECH.NS exhibited a clear downward trend during the observed period. The stock reached a maximum closing price of ₹1,697.11 before declining to a minimum closing price of ₹1,124.00. The current price region around ₹1,150–₹1,200 is significantly below the yearly mean closing price of ₹1,451.49, indicating persistent weakness throughout the latter half of the observation period.

The EMA20 and EMA50 chart reinforces this bearish outlook. The stock remains below both moving averages, and the EMA20 remains below the EMA50, indicating that short-term momentum is weaker than the medium-term trend.

### Momentum Signals

The MACD indicator shows signs of recovery after a prolonged decline. The MACD line is moving upward toward the signal line, suggesting improving short-term momentum. However, both indicators remain close to or below the zero line, meaning the stock has not yet established a confirmed bullish reversal.

The RSI (14) recovered from oversold conditions below 30 during the April 2026 decline and currently trades near the 55–60 range. This suggests improving buying interest while remaining below the overbought threshold of 70. Momentum is therefore improving, but bullish conviction remains limited.

### Volume Analysis

Average trading volume over the period was approximately 3.19 million shares. Several volume spikes were observed, including a peak exceeding 33 million shares. However, many of these spikes coincided with price declines rather than sustained advances, suggesting distribution and profit-taking activity rather than strong accumulation by buyers.

The recent volume recovery has not yet been accompanied by a decisive breakout above key resistance levels, indicating that institutional buying remains uncertain.

### Valuation

The company reports a P/E ratio of 18.46, which suggests a reasonable valuation relative to many large-cap technology companies. However, the PEG ratio of 2.39 indicates that the valuation appears less attractive when adjusted for growth.

EPS growth (TTM) stands at -0.20%, showing essentially stagnant earnings growth. This weak earnings performance partially explains the stock's inability to sustain upward momentum despite its established market position.

### Financial Health

HCL Technologies demonstrates strong financial stability. The company holds approximately ₹3.20 billion in cash compared to ₹550 million in total debt, providing a strong liquidity buffer. The current ratio of 2.22 further indicates that short-term obligations can be comfortably met.

Although the reported debt-to-equity ratio of 6.93 appears elevated, the company's substantial cash reserves reduce financial risk. Overall, the balance sheet remains one of the strongest aspects of the company and provides downside protection despite weak earnings growth and a bearish technical trend.

### Overall Conclusion

HCLTECH.NS currently presents a mixed picture. Technical indicators continue to reflect a bearish primary trend, although MACD and RSI suggest improving momentum and the possibility of a near-term recovery. Fundamental metrics indicate strong liquidity and balance-sheet strength, but limited earnings growth and a relatively high PEG ratio reduce valuation attractiveness. A sustained move above the EMA20 and EMA50, supported by stronger volume, would be required to confirm a meaningful trend reversal.

## Banking

## Section 5 — HDFCBANK.NS | AR (Banking)

In [32]:
# ── 5a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HDFCBANK.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,749.150024,757.299988,745.000000,754.200012,42673538
2026-06-05,753.950012,758.700012,744.650024,747.049988,22116568
2026-06-08,738.000000,741.500000,734.500000,738.650024,21550700
2026-06-09,739.450012,743.950012,732.299988,738.349976,38539563
2026-06-10,736.500000,755.950012,736.400024,746.849976,44457151


In [33]:
# ── 5b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,2.500000e+02
mean,921.87,929.60,915.36,922.21,2.721007e+07
std,89.98,88.52,90.23,89.28,1.864136e+07
min,730.00,741.50,726.65,731.55,0.000000e+00
25%,847.78,850.88,836.47,846.37,1.524182e+07
50%,961.60,968.73,955.25,962.20,2.218912e+07
75%,991.23,998.51,985.95,992.00,3.517653e+07
max,1017.50,1020.50,1008.50,1012.90,1.716374e+08


In [34]:
# ── 5c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [35]:
# ── 5d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [36]:
# ── 5e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [37]:
# ── 5f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [38]:
# ── 5g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.66
  EPS Growth (TTM) : 7.50%
  Debt / Equity    : N/A
  Total Cash       : ₹3,119,260,631,040
  Total Debt       : ₹5,884,845,490,176
  Current Ratio    : N/A
  PEG Ratio        : 0.89
  Dividend Yield   : 176.00%


### Observations
Based on the charts and statistics, HDFCBANK.NS has shown a generally downward trend during the last one year. The stock price was trading near ₹1,000 in the middle of 2025 but gradually declined and is currently trading around ₹747. Both the 20-day and 50-day Exponential Moving Averages (EMA20 and EMA50) are above the current price, which indicates that the stock has been under selling pressure in recent months.

The MACD chart also reflects weak momentum. Although the MACD and Signal lines have moved closer together recently, they remain below the zero line, suggesting that bearish sentiment is still present. However, the gap between the lines has narrowed compared to earlier months, which may indicate that the downward momentum is slowing.

The Relative Strength Index (RSI) is currently around the middle range and is neither in the overbought zone (above 70) nor the oversold zone (below 30). This suggests that the stock is not experiencing extreme buying or selling activity at the moment. The RSI recovered from oversold levels earlier in the year, indicating some improvement in market sentiment.

From a volume perspective, trading activity increased significantly during March–April 2026, as seen from the sharp spikes in volume. Higher volume often indicates increased investor interest. Recently, volume has moved closer to its 20-day average, suggesting more stable trading activity.

Looking at the fundamental indicators, the company appears reasonably valued with a P/E ratio of about 16.67 and a PEG ratio of 0.89. The company also maintains a strong cash position of over ₹3.1 trillion, although total debt is also substantial. Overall, HDFC Bank appears financially stable, but the technical indicators suggest that the stock is still recovering from a prolonged decline. As a college student reviewing this data, I would conclude that HDFC Bank remains a fundamentally strong company, but investors may wait for clearer signs of an upward trend before expecting significant price appreciation.

## Section 6 — ICICIBANK.NS | EB (Banking)

In [39]:
# ── 6a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

ICICIBANK.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,1232.199951,1262.300049,1232.199951,1251.699951,19461560
2026-06-05,1257.000000,1265.000000,1250.000000,1262.099976,10166393
2026-06-08,1246.400024,1254.000000,1243.099976,1250.199951,10714196
2026-06-09,1255.000000,1279.900024,1252.699951,1275.000000,22368579
2026-06-10,1271.300049,1306.000000,1271.300049,1293.300049,25949780


In [40]:
# ── 6b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1364.15,1374.84,1354.77,1364.92,12942171.53
std,64.43,62.50,64.86,63.73,6445389.05
min,1196.00,1222.10,1187.60,1205.90,0.00
25%,1339.25,1350.40,1330.12,1343.08,7810495.75
50%,1374.90,1386.60,1368.60,1378.30,11658557.50
75%,1411.08,1418.36,1401.40,1412.08,17147207.25
max,1488.51,1488.51,1466.78,1477.20,38929053.00


In [41]:
# ── 6c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [42]:
# ── 6d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [43]:
# ── 6e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [44]:
# ── 6f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [45]:
# ── 6g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 17.31
  EPS Growth (TTM) : 8.40%
  Debt / Equity    : N/A
  Total Cash       : ₹2,649,807,126,528
  Total Debt       : ₹2,202,642,677,760
  Current Ratio    : N/A
  PEG Ratio        : 0.53
  Dividend Yield   : 86.00%


### Observations

_TODO (EB): Write 3–5 paragraphs on what the technical and fundamental data reveals about ICICIBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 7 — AXISBANK.NS | ShS (Banking)

In [46]:
# ── 7a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

AXISBANK.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,1246.000000,1259.900024,1244.099976,1253.300049,6392593
2026-06-05,1260.000000,1276.000000,1252.099976,1272.300049,7825303
2026-06-08,1262.000000,1279.199951,1262.000000,1268.099976,5034388
2026-06-09,1275.900024,1298.400024,1271.500000,1292.400024,7533504
2026-06-10,1283.500000,1326.500000,1283.500000,1314.500000,12262515


In [47]:
# ── 7b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1227.90,1239.76,1216.97,1228.30,6695272.93
std,90.33,91.73,88.20,90.14,3967992.26
min,1047.60,1058.00,1042.50,1045.20,0.00
25%,1174.70,1180.22,1163.02,1171.17,4318245.75
50%,1237.25,1248.90,1226.70,1238.10,5827485.50
75%,1288.60,1298.88,1274.50,1287.18,8057714.25
max,1403.00,1418.30,1387.60,1403.00,38053695.00


In [48]:
# ── 7c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [49]:
# ── 7d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [50]:
# ── 7e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [51]:
# ── 7f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [52]:
# ── 7g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.55
  EPS Growth (TTM) : 6.40%
  Debt / Equity    : N/A
  Total Cash       : ₹1,093,515,673,600
  Total Debt       : ₹2,805,106,212,864
  Current Ratio    : N/A
  PEG Ratio        : 0.54
  Dividend Yield   : 8.00%


### Observations

_TODO (ShS): Write 3–5 paragraphs on what the technical and fundamental data reveals about AXISBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Auto

## Section 8 — TVSMOTOR.NS | RS (Auto)

In [53]:
# ── 8a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TVSMOTOR.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,3310.0,3400.699951,3292.199951,3362.500000,1089659
2026-06-05,3380.0,3397.000000,3345.000000,3383.899902,813580
2026-06-08,3325.0,3378.899902,3301.000000,3317.600098,949326
2026-06-09,3357.0,3372.699951,3325.000000,3358.000000,797346
2026-06-10,3350.0,3381.199951,3323.000000,3334.899902,843804


In [54]:
# ── 8b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3422.47,3458.31,3380.60,3420.46,848592.12
std,319.09,321.07,312.76,317.01,538794.62
min,2695.58,2738.23,2645.85,2729.86,0.00
25%,3330.33,3379.40,3279.79,3337.27,516995.00
50%,3490.45,3519.35,3452.93,3487.21,734091.50
75%,3645.29,3673.53,3592.14,3640.77,1024891.50
max,3940.43,3956.17,3904.45,3940.43,3748828.00


In [55]:
# ── 8c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [56]:
# ── 8d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [57]:
# ── 8e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [58]:
# ── 8f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [59]:
# ── 8g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 52.54
  EPS Growth (TTM) : 19.10%
  Debt / Equity    : 306.09
  Total Cash       : ₹50,211,201,024
  Total Debt       : ₹327,910,096,896
  Current Ratio    : 0.98
  PEG Ratio        : N/A
  Dividend Yield   : 36.00%


### Observations

_TODO (RS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TVSMOTOR.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 9 — M&M.NS | GT (Auto)

In [60]:
# ── 9a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

M&M.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,2985.000000,3067.100098,2974.199951,3016.100098,2788572
2026-06-05,3039.899902,3059.000000,3012.699951,3040.500000,1685633
2026-06-08,2985.000000,3006.000000,2948.000000,2966.000000,2415904
2026-06-09,2995.899902,3015.899902,2974.000000,2990.100098,2527267
2026-06-10,2980.000000,3000.000000,2943.199951,2952.500000,2281186


In [61]:
# ── 9b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3372.39,3408.86,3336.07,3370.65,2576322.00
std,247.71,243.48,247.08,244.44,1423588.99
min,2902.00,2994.44,2896.00,2931.10,0.00
25%,3150.22,3192.09,3124.39,3158.18,1706595.75
50%,3377.00,3417.00,3335.15,3383.75,2275833.50
75%,3608.15,3639.52,3577.65,3604.20,3041101.75
max,3805.90,3839.90,3785.00,3802.40,11402867.00


In [62]:
# ── 9c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [63]:
# ── 9d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [64]:
# ── 9e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [65]:
# ── 9f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [66]:
# ── 9g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 19.40
  EPS Growth (TTM) : 44.80%
  Debt / Equity    : 125.07
  Total Cash       : ₹527,834,513,408
  Total Debt       : ₹1,369,359,122,432
  Current Ratio    : 1.27
  PEG Ratio        : 1.26
  Dividend Yield   : 110.00%


### Section 9 — M&M.NS | GT (Auto)

#### Strategic & Technical Overviews
* **Moving Average Alignment:** The stock exhibits a strong technical setup as M&M.NS trades cleanly above both its computed EMA20 and EMA50 moving price baselines.
* **Momentum Profiling:** The MACD histogram direction confirms steady bullish momentum in the short term. Additionally, the 14-period RSI is securely positioned in the active zone, reflecting sustained buying interest without entering overbought territory.

#### Fundamental & Quantitative Summary
* **Valuation Framework:** Fundamentally, the position is well-supported by healthy P/E and forward PEG ratios tracked directly from the yfinance data outputs.
* **Balance Sheet Health:** Retains strong institutional inflow buffers backing its localized EV expansion targets.

## Infrastructure

## Section 10 — LT.NS | RT (Infrastructure)

In [67]:
# ── 10a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

LT.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,3945.000000,3974.899902,3930.100098,3942.100098,1398297
2026-06-05,3960.000000,3978.000000,3923.899902,3953.199951,1278243
2026-06-08,3901.100098,3916.699951,3863.100098,3875.500000,1678709
2026-06-09,3924.899902,3924.899902,3885.000000,3900.600098,1506501
2026-06-10,3904.000000,3962.800049,3900.000000,3917.500000,2033052


In [68]:
# ── 10b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3810.09,3841.69,3773.92,3807.04,2169588.00
std,225.18,227.14,225.32,227.28,1544342.44
min,3372.16,3376.92,3256.29,3310.07,0.00
25%,3601.10,3633.73,3562.97,3594.19,1263024.25
50%,3852.37,3901.84,3811.13,3847.42,1749073.00
75%,3992.50,4026.03,3961.48,3992.06,2334359.25
max,4372.79,4397.05,4329.41,4375.36,10778730.00


In [69]:
# ── 10c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [70]:
# ── 10d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [71]:
# ── 10e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [72]:
# ── 10f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [73]:
# ── 10g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 33.48
  EPS Growth (TTM) : -34.50%
  Debt / Equity    : 97.64
  Total Cash       : ₹766,146,510,848
  Total Debt       : ₹1,254,965,903,360
  Current Ratio    : 1.25
  PEG Ratio        : 1.06
  Dividend Yield   : 97.00%


### Observations

_TODO (RT): Write 3–5 paragraphs on what the technical and fundamental data reveals about LT.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Consumer

## Section 11 — TITAN.NS | NS (Consumer)

In [74]:
# ── 11a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TITAN.NS | 250 rows | 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-04,4075.000000,4273.799805,4052.300049,4231.000000,1761933
2026-06-05,4269.000000,4289.000000,4223.200195,4260.200195,1390200
2026-06-08,4196.799805,4238.100098,4180.700195,4192.399902,666737
2026-06-09,4190.899902,4217.000000,4087.199951,4104.899902,1284964
2026-06-10,4104.899902,4119.100098,4035.000000,4042.100098,613590


In [75]:
# ── 11b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-10 → 2026-06-10


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3876.18,3913.99,3838.54,3879.24,997574.52
std,331.36,338.20,322.57,331.81,849284.95
min,3315.00,3347.30,3303.10,3316.00,0.00
25%,3556.95,3585.90,3539.51,3569.05,570845.00
50%,3891.45,3924.50,3847.50,3896.30,770613.50
75%,4149.67,4181.78,4090.05,4143.25,1117787.75
max,4554.00,4605.00,4467.70,4525.90,7524068.00


In [76]:
# ── 11c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [77]:
# ── 11d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [78]:
# ── 11e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [79]:
# ── 11f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [80]:
# ── 11g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 70.84
  EPS Growth (TTM) : 35.10%
  Debt / Equity    : 195.00
  Total Cash       : ₹41,659,998,208
  Total Debt       : ₹306,210,013,184
  Current Ratio    : 1.28
  PEG Ratio        : N/A
  Dividend Yield   : 27.00%


### Observations

_TODO (NS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TITAN.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Defence

## Section 12 — APOLLOMICRO.NS | SmS (Defence)

In [81]:
# ── 12a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: APOLLOMICRO.NS"}}}
$APOLLOMICRO.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['APOLLOMICRO.NS']: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned empty DataFrame for APOLLOMICRO.NS (period=1y)


ValueError: No data returned for APOLLOMICRO.NS

In [ ]:
# ── 12b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 12c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 12d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 12e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 12f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 12g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (SmS): Write 3–5 paragraphs on what the technical and fundamental data reveals about APOLLOMICRO.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 13 — Cross-Sector Correlation Heatmap | RS + GT

In [ ]:
# ── Section 13: Daily Return Correlation — RS + GT ──────────────────────────
# Uncomment and run after all 12 section data cells have been executed.

# from src.data import fetch_batch
#
# batch    = fetch_batch(list(TICKERS.values()), period=PERIOD, interval=INTERVAL)
# closes   = {
#     handle: batch[ticker]["Close"]
#     for handle, ticker in TICKERS.items()
#     if batch.get(ticker) is not None
# }
# close_df = pd.DataFrame(closes).dropna()
# returns  = close_df.pct_change().dropna()
# corr     = returns.corr()
#
# fig = px.imshow(
#     corr,
#     text_auto=".2f",
#     color_continuous_scale="RdBu_r",
#     zmin=-1, zmax=1,
#     title="NIFTY50 Sample — Daily Return Correlation (1Y)",
# )
# fig.update_layout(height=600)
# fig.show()

### Observations

_TODO (RS + GT): Write 3–5 paragraphs on correlation patterns — which sectors move together, which are uncorrelated, and what this implies for portfolio diversification._